# Лабораторная работа 6. Проверка автокорреляции и сдвига дисперсии

**Тема:** проверка гипотезы об отсутствии автокорреляции (критерий по выборочному коэффициенту автокорреляции и критерий Морана) и проверка стабильности дисперсии во времени по критерию Хсу.

## Постановка задачи
1. Сгенерировать четыре временных ряда: исходную независимую выборку $X_1$ и три ряда $X_2, X_3, X_4$ с искусственно введённой автокорреляционной зависимостью. Проверить наличие автокорреляции на шагах (лагах) 1, 2, 3 по двум критериям: базовому (по коэффициенту $r$) и критерию Морана. Уровень значимости $lpha = 0.05$.
2. По критерию Хсу проверить гипотезу об отсутствии сдвига дисперсии для однородной выборки и для выборок с искусственно увеличенной дисперсией на части ряда.

In [1]:
# Импорт библиотек и настройка
import numpy as np
from scipy import stats

np.random.seed(42)
alpha = 0.05
n = 500

## 1. Проверка автокорреляции

### 1.1 Генерация выборок $X_1$–$X_4$

In [2]:
# 1.1.1 Исходная выборка
X1 = np.random.normal(6, 4, n)

# 1.1.2 X2j = 2*X1j - X1,j-1 + ε, циклически
X2 = np.zeros(n)
eps = np.random.normal(0, 1, n)
X2[0] = 2 * X1[0] - X1[-1] + eps[0]
for j in range(1, n):
    X2[j] = 2 * X1[j] - X1[j-1] + eps[j]

# 1.1.3 X3j = X1j + 0.1*X1,j-1 + 0.1*ε
X3 = np.zeros(n)
X3[0] = X1[0] + 0.1 * X1[-1] + 0.1 * eps[0]
for j in range(1, n):
    X3[j] = X1[j] + 0.1 * X1[j-1] + 0.1 * eps[j]

# 1.1.4 X4j = 2*X1j - 0.3*X4,j-1 + 0.1*ε, X0=0
X4 = np.zeros(n)
X4[0] = 2 * X1[0] + 0.1 * eps[0]
for j in range(1, n):
    X4[j] = 2 * X1[j] - 0.3 * X4[j-1] + 0.1 * eps[j]

### 1.2 Функции критериев: коэффициент автокорреляции, базовый критерий и критерий Морана

In [3]:
# ====================== Коэффициент автокорреляции ======================
def autocorr_coeff(x, lag=1):
    """Выборочный коэффициент автокорреляции с заданным шагом."""
    x1 = x[:-lag]
    x2 = x[lag:]
    return np.corrcoef(x1, x2)[0, 1]


# ====================== 1.2.1 Критерий отсутствия автокорреляции ======================
def autocorr_test(x, lag=1):
    """Базовый критерий по коэффициенту автокорреляции: при H0 z = sqrt(n)*r ~ N(0,1)."""
    n = len(x)
    r = autocorr_coeff(x, lag)
    z = np.sqrt(n) * r
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p_value


# ====================== 1.2.2 Критерий Морана ======================
def moran_statistic(x, lag=1):
    """Статистика Морана r_{1,n}^M (преобразование коэффициента r)."""
    n = len(x)
    r = autocorr_coeff(x, lag)
    # Формула (2.4)
    r_M = np.sqrt(n - 1) * (n * r + 1) / (n - 2)
    return r_M


def moran_test(x, lag=1):
    r_M = moran_statistic(x, lag)
    # По методичке при n > 50 ≈ N(0,1)
    p_value = 2 * (1 - stats.norm.cdf(abs(r_M)))
    return r_M, p_value

### 1.3 Выполнение тестов на автокорреляцию (лаги 1, 2, 3)

In [4]:
# ====================== Выполнение тестов ======================
samples = [
    ('X1 ', X1),
    ('X2 ', X2),
    ('X3 ', X3),
    ('X4 ', X4)
]

print("=== 1.2.1 Критерий отсутствия автокорреляции (по коэффициенту r) ===\n")
for name, data in samples:
    print(f"--- {name} ---")
    for lag in [1, 2, 3]:
        z, p = autocorr_test(data, lag)
        conclusion = "Отвергаем H0 (есть автокорреляция)" if p < alpha else "Принимаем H0 (нет автокорреляции)"
        print(f"  Lag {lag:2d}:  z   = {z:8.4f}    p-value = {p:.4f}  →  {conclusion}")
    print("-" * 70)

print("\n=== 1.2.2 Критерий Морана ===\n")
for name, data in samples:
    print(f"--- {name} ---")
    for lag in [1, 2, 3]:
        r_M, p = moran_test(data, lag)
        conclusion = "Отвергаем H0 (есть автокорреляция)" if p < alpha else "Принимаем H0 (нет автокорреляции)"
        print(f"  Lag {lag:2d}:  r_M = {r_M:8.4f}    p-value = {p:.4f}  →  {conclusion}")
    print("-" * 70)

=== 1.2.1 Критерий отсутствия автокорреляции (по коэффициенту r) ===

--- X1  ---
  Lag  1:  z   =  -0.0898    p-value = 0.9285  →  Принимаем H0 (нет автокорреляции)
  Lag  2:  z   =  -0.1917    p-value = 0.8480  →  Принимаем H0 (нет автокорреляции)
  Lag  3:  z   =   0.3010    p-value = 0.7634  →  Принимаем H0 (нет автокорреляции)
----------------------------------------------------------------------
--- X2  ---
  Lag  1:  z   =  -8.8884    p-value = 0.0000  →  Отвергаем H0 (есть автокорреляция)
  Lag  2:  z   =  -0.2950    p-value = 0.7680  →  Принимаем H0 (нет автокорреляции)
  Lag  3:  z   =   1.0102    p-value = 0.3124  →  Принимаем H0 (нет автокорреляции)
----------------------------------------------------------------------
--- X3  ---
  Lag  1:  z   =   2.0806    p-value = 0.0375  →  Отвергаем H0 (есть автокорреляция)
  Lag  2:  z   =  -0.1848    p-value = 0.8534  →  Принимаем H0 (нет автокорреляции)
  Lag  3:  z   =   0.1194    p-value = 0.9050  →  Принимаем H0 (нет автокоррел

## 2. Критерий Хсу: проверка сдвига дисперсии

In [5]:
n_hsu = 600
X_hsu = np.random.normal(0, 5, n_hsu)

def hsu_statistic(x):
    """Статистика критерия Хсу D = max|S_k/S_n - k/n|."""
    n = len(x)
    x_mean = np.mean(x)
    cum_sq = np.cumsum((x - x_mean)**2)
    ratios = cum_sq[:-1] / cum_sq[-1]
    k = np.arange(1, n)
    D = np.max(np.abs(ratios - k / n))
    return D

def hsu_test(x):
    n = len(x)
    D = hsu_statistic(x)
    H = np.sqrt(n) * D                 # статистика D, умноженная на корень из n
    p_value = stats.kstwobign.sf(H)    # P(K >= H), распределение Колмогорова
    return D, H, p_value


print("2.1 Оригинальная выборка - N(0,5):")
D1, H1, p1 = hsu_test(X_hsu)
print(f"   D = {D1:.5f}, H = sqrt(n)*D = {H1:.4f}, p = {p1:.4f} → {'Сдвиг есть' if p1 < alpha else 'Сдвига нет'}")

X_shift1 = X_hsu.copy()
X_shift1[n_hsu//2:] *= 2
D2, H2, p2 = hsu_test(X_shift1)
print(f"\n2.2 Вторая половина × 2:")
print(f"   D = {D2:.5f}, H = sqrt(n)*D = {H2:.4f}, p = {p2:.4f} → {'Сдвиг обнаружен' if p2 < alpha else 'Сдвига нет'}")

X_shift2 = X_hsu.copy()
third = n_hsu // 3
X_shift2[-third:] *= 1.2
D3, H3, p3 = hsu_test(X_shift2)
print(f"\n2.3 Последняя треть × 1.2:")
print(f"   D = {D3:.5f}, H = sqrt(n)*D = {H3:.4f}, p = {p3:.4f} → {'Сдвиг обнаружен' if p3 < alpha else 'Сдвига нет'}")

2.1 Оригинальная выборка - N(0,5):
   D = 0.03367, H = sqrt(n)*D = 0.8248, p = 0.5043 → Сдвига нет

2.2 Вторая половина × 2:
   D = 0.31095, H = sqrt(n)*D = 7.6166, p = 0.0000 → Сдвиг обнаружен

2.3 Последняя треть × 1.2:
   D = 0.10082, H = sqrt(n)*D = 2.4696, p = 0.0000 → Сдвиг обнаружен


## Выводы

1. **Автокорреляция.** Для исходной независимой выборки $X_1$ оба критерия не обнаруживают автокорреляции ни на одном лаге. Для рядов $X_2, X_3, X_4$, построенных с зависимостью от предыдущих значений, на лаге 1 гипотеза об отсутствии автокорреляции уверенно отвергается, а на лагах 2 и 3 зависимость уже не выявляется — введённая связь является корреляцией первого порядка.
2. **Критерий Морана** даёт практически те же значения и выводы, что и базовый критерий по коэффициенту $r$ (при $n > 50$ его статистика близка к стандартной нормальной), и служит уточнённой версией критерия автокорреляции.
3. **Критерий Хсу.** Для однородной выборки $N(0,5)$ сдвиг дисперсии не обнаруживается. При удвоении дисперсии на второй половине ряда и при увеличении дисперсии на последней трети критерий уверенно фиксирует сдвиг (p-value $pprox 0$), то есть критерий чувствителен к изменению дисперсии во времени.